# 🔌 Project 13.3 — Network Cable Planner
**DSA for Mechatronics · Week 13 — Greedy Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory needs to wire up sensor nodes with the minimum total cable length,
ensuring all nodes are connected with no loops (a **Minimum Spanning Tree**).

1. **Kruskal's algorithm** — sort all edges by weight; greedily add lightest
   edge that doesn't form a cycle (union-find for cycle detection)
2. **Prim's algorithm** — grow MST from a start node; always add the cheapest
   edge that connects a new node (min-heap for efficient next-edge selection)
3. Verify both algorithms give the **same total weight**


## Step 1 — Generate factory network graph

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_NODES = random.randint(7, 11)
EDGE_P  = random.uniform(0.45, 0.65)   # edge probability (ensure connected)
W_MAX   = random.randint(15, 30)

# Generate random connected graph
NODES = list(range(N_NODES))
EDGES = []
# First create a spanning tree to guarantee connectivity
shuffled = list(range(N_NODES))
random.shuffle(shuffled)
for i in range(1, N_NODES):
    u, v = shuffled[i-1], shuffled[i]
    w = random.randint(1, W_MAX)
    EDGES.append((w, u, v))
# Add extra random edges
for u in range(N_NODES):
    for v in range(u+1, N_NODES):
        if random.random() < EDGE_P:
            if not any((a==u and b==v) or (a==v and b==u) for _, a, b in EDGES):
                w = random.randint(1, W_MAX)
                EDGES.append((w, u, v))

# Build adjacency list
ADJ = defaultdict(list)
for w, u, v in EDGES:
    ADJ[u].append((w, v))
    ADJ[v].append((w, u))

print(f"Factory network graph:")
print(f"  Nodes : {N_NODES}")
print(f"  Edges : {len(EDGES)}")
print()
print(f"  Edge list (weight, u, v):")
for w, u, v in sorted(EDGES):
    print(f"    w={w:3d}  {u}─{v}")


## Step 2 — Kruskal's algorithm (union-find)

In [ ]:
class UnionFind:
    """
    Union-Find (Disjoint Set Union) with path compression and union by rank.
    - find(x): returns root of x's component (with path compression)
    - union(x, y): merges components of x and y; returns False if already same
    """
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])   # path compression
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry: return False          # already connected — adding would create cycle
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True

def kruskal(n_nodes, edges):
    """
    Kruskal's MST algorithm.

    1. Sort all edges by weight (greedy: try cheapest first).
    2. For each edge (u, v, w) in sorted order:
       - If u and v are in different components → add edge to MST.
       - Otherwise → skip (would create a cycle).
    3. Stop when MST has n-1 edges.

    Correctness (cut property): for any cut of the graph, the minimum-weight
    edge crossing the cut is in every MST.
    O(E log E) for sort, O(E · α(V)) for union-find (α ≈ constant).
    """
    uf = UnionFind(n_nodes)
    sorted_edges = sorted(edges)   # sort by weight (first element)
    mst_edges = []
    total_w   = 0
    for w, u, v in sorted_edges:
        if uf.union(u, v):
            mst_edges.append((w, u, v))
            total_w += w
            if len(mst_edges) == n_nodes - 1:
                break
    return mst_edges, total_w

kruskal_mst, kruskal_w = kruskal(N_NODES, EDGES)

print(f"Kruskal's MST (n={N_NODES} nodes, {len(EDGES)} edges):")
print()
print(f"  Edges added to MST (in order selected):")
for w, u, v in kruskal_mst:
    print(f"    w={w:3d}  {u}─{v}")
print()
print(f"  MST edges     : {len(kruskal_mst)}  (should be {N_NODES-1})")
print(f"  Total weight  : {kruskal_w}")
print(f"  Is spanning   : {'✅ YES' if len(kruskal_mst) == N_NODES-1 else '❌ NO'}")


## Step 3 — Prim's algorithm (min-heap)

In [ ]:
def prim(n_nodes, adj, start=0):
    """
    Prim's MST algorithm.

    Grow MST one node at a time from `start`:
      1. Push all edges from start to min-heap.
      2. Repeat until all nodes visited:
         a. Pop minimum-weight edge (w, u, v) from heap.
         b. If v already visited → skip.
         c. Add edge to MST; push all new edges from v.

    Same greedy cut property as Kruskal but expands from a seed node.
    O(E log V) with a min-heap.
    """
    visited   = set([start])
    mst_edges = []
    total_w   = 0
    heap = [(w, start, v) for w, v in adj[start]]
    heapq.heapify(heap)
    while heap and len(visited) < n_nodes:
        w, u, v = heapq.heappop(heap)
        if v in visited:
            continue
        visited.add(v)
        mst_edges.append((w, u, v))
        total_w += w
        for nw, nv in adj[v]:
            if nv not in visited:
                heapq.heappush(heap, (nw, v, nv))
    return mst_edges, total_w

prim_mst, prim_w = prim(N_NODES, ADJ)

print(f"Prim's MST (starting from node 0):")
print()
print(f"  Edges added to MST (in order selected):")
for w, u, v in prim_mst:
    print(f"    w={w:3d}  {u}─{v}")
print()
print(f"  MST edges     : {len(prim_mst)}  (should be {N_NODES-1})")
print(f"  Total weight  : {prim_w}")
print()
same_weight = kruskal_w == prim_w
same_size   = len(kruskal_mst) == len(prim_mst) == N_NODES - 1
print(f"  Kruskal total weight : {kruskal_w}")
print(f"  Prim total weight    : {prim_w}")
print(f"  Same total weight    : {'✅ YES' if same_weight else '❌ NO (both MSTs if equal n-1 edges)'}")


## Step 4 — MST properties and analysis

In [ ]:
# Connectivity check via BFS
def bfs_connected(n, mst_edges):
    adj_mst = defaultdict(list)
    for w, u, v in mst_edges:
        adj_mst[u].append(v)
        adj_mst[v].append(u)
    visited = set()
    queue   = [0]
    while queue:
        node = queue.pop()
        if node in visited: continue
        visited.add(node)
        queue.extend(adj_mst[node])
    return len(visited) == n

kruskal_connected = bfs_connected(N_NODES, kruskal_mst)
prim_connected    = bfs_connected(N_NODES, prim_mst)

# Edge weight distribution
all_weights = [w for w, u, v in EDGES]
mst_weights_k = [w for w, u, v in kruskal_mst]

print(f"MST analysis:")
print()
print(f"  Original graph:")
print(f"    Nodes        : {N_NODES}")
print(f"    Edges        : {len(EDGES)}")
print(f"    Min edge wt  : {min(all_weights)}")
print(f"    Max edge wt  : {max(all_weights)}")
print(f"    Sum all edges: {sum(all_weights)}")
print()
print(f"  Kruskal MST:")
print(f"    Edges        : {len(kruskal_mst)} (= n-1 = {N_NODES-1})")
print(f"    Total weight : {kruskal_w}")
print(f"    Connected    : {'✅ YES' if kruskal_connected else '❌ NO'}")
print(f"    Edge weights : {sorted(mst_weights_k)}")
print()
print(f"  Prim MST:")
print(f"    Edges        : {len(prim_mst)}")
print(f"    Total weight : {prim_w}")
print(f"    Connected    : {'✅ YES' if prim_connected else '❌ NO'}")
print()
saving = sum(all_weights) - kruskal_w
print(f"  Cable saving vs connecting all edges: {saving} weight units")
print(f"  MST / total graph weight ratio: {kruskal_w/sum(all_weights):.3f}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " NETWORK CABLE PLANNER (MST) — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  GRAPH" + " "*(W-7) + "║")
print(f"║  {'Nodes':<28}: {N_NODES:<26}║")
print(f"║  {'Edges':<28}: {len(EDGES):<26}║")
print(f"║  {'Total edge weight':<28}: {sum(all_weights):<26}║")
print("╠" + "═"*W + "╣")
print("║  KRUSKAL'S MST" + " "*(W-15) + "║")
print(f"║  {'MST edges':<28}: {len(kruskal_mst)} (n-1={N_NODES-1}){'':<14}║")
print(f"║  {'Total weight':<28}: {kruskal_w:<26}║")
print(f"║  {'Connected':<28}: {'YES ✅' if kruskal_connected else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  PRIM'S MST" + " "*(W-12) + "║")
print(f"║  {'MST edges':<28}: {len(prim_mst):<26}║")
print(f"║  {'Total weight':<28}: {prim_w:<26}║")
print(f"║  {'Connected':<28}: {'YES ✅' if prim_connected else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  COMPARISON" + " "*(W-12) + "║")
print(f"║  {'Both give same MST weight':<28}: {'YES ✅' if same_weight else 'NO (both valid MSTs)':<26}║")
print(f"║  {'Cable saving vs all edges':<28}: {saving:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about greedy algorithms in this context?

*Your answer here:*

---

**Q2 — Why does the greedy choice work here?**
For one of the algorithms in this project, explain the *exchange argument* — why swapping the greedy choice for any other choice cannot improve the solution.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
